In [ ]:
import json
import kagglehub
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from ast import literal_eval
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report

from open_video_summary.adapters.llm import OllamaAdapter
from open_video_summary.utils.config import PROJECT_DIR
from open_video_summary.core.segmenter.prompts import VideoSegmenterPrompts

def show_results(df_results: pd.DataFrame):
    # Calculate classification metrics
    report = classification_report(df_results['class'], df_results['prediction'], output_dict=True)
    metrics_df = pd.DataFrame(report).transpose()

    # Print precision, recall, and F1-score
    print(metrics_df[['precision', 'recall', 'f1-score']])

    # Plot precision, recall, and F1-score
    metrics_df = metrics_df[:-3]  # Exclude 'accuracy', 'macro avg', and 'weighted avg'
    metrics_df[['precision', 'recall', 'f1-score']].plot(kind='bar', figsize=(10, 6))
    plt.title('Precision, Recall, and F1-Score per Class')
    plt.xlabel('Class')
    plt.ylabel('Score')
    plt.xticks(rotation=45)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

# Text Classification

## text-classification-documentation

In [ ]:
dataset_path = Path(kagglehub.dataset_download("tanishqdublish/text-classification-documentation", force_download=True))

destination = Path((PROJECT_DIR / "data/datasets/text-classification-documentation"))
destination.mkdir(parents=True, exist_ok=True)

for file in dataset_path.iterdir():
    print(file)
    if file.is_file():
        print(file.name)
        file.rename(destination / file.name)

dataset_path = Path(kagglehub.dataset_download("saurabhshahane/ecommerce-text-classification"))
destination = Path((PROJECT_DIR / "data/datasets/ecommerce-text-classification"))
destination.mkdir(parents=True, exist_ok=True)

for file in dataset_path.iterdir():
    print(file)
    if file.is_file():
        print(file.name)
        file.rename(destination / file.name)

In [ ]:
df = pd.read_csv((PROJECT_DIR / "data/datasets/text-classification-documentation/df_file.csv"))
topics = {
    "0": "Politics",
    "1": "Sport",
    "2": "Technology",
    "3": "Entertainment",
    "4": "Business",
}

df = df.rename(columns={"Text": "content", "Label": "topic"})
df = df.sample(frac=1)
df.head()

In [ ]:
results_path = (PROJECT_DIR / "data/datasets/text-classification-documentation/results.csv")

prompts_template = VideoSegmenterPrompts()
llm_adapter = OllamaAdapter()

with open(results_path, "w") as f:
    f.write("index,class,prediction\n")

for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Classifying content"):
    prompt = prompts_template.classify_subtopic.format(
        content=row["content"], topics=topics
    )

    topics_str = llm_adapter.generate_pattern(
        prompt=prompt,
        pattern="(\{.*?\})",
        options={"format": "json", "temperature": 0.2},
    )
    cluster_topic = literal_eval(topics_str)
    prediction, _ = cluster_topic.popitem()

    with open(results_path, "a") as f:
        f.write(f"{index},{row['topic']},{int(prediction)}\n")

In [ ]:
results_path = (PROJECT_DIR / "data/datasets/text-classification-documentation/results.csv")

df_results = pd.read_csv(results_path)
show_results(df_results)

## ecommerceDataset

In [ ]:
df = pd.read_csv((PROJECT_DIR / "data/datasets/ecommerce-text-classification/ecommerceDataset.csv"), header=None, names=["Label", "Text"])
topics = {
    "0": "Books",
    "1": "Household",
    "2": "Clothing & Accessories",
    "3": "Electronics",
}

df = df.rename(columns={"Text": "content", "Label": "topic"})
df["topic"] = df["topic"].map({v: k for k, v in topics.items()})
df = df.sample(frac=1)
print(len(df))
df.head()

In [ ]:
results_path = (PROJECT_DIR / "data/datasets/ecommerce-text-classification/results.csv")

df_results = pd.read_csv(results_path)
df_results.set_index("index", inplace=True)

df = df[~df.index.isin(df_results.index)]
print(len(df))
df.head()

In [ ]:
prompts_template = VideoSegmenterPrompts()
llm_adapter = OllamaAdapter()

# with open(results_path, "w") as f:
#     f.write("index,class,prediction\n")

for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Classifying content"):
    prompt = prompts_template.classify_subtopic.format(
        content=row["content"], topics=topics
    )

    topics_str = llm_adapter.generate_pattern(
        prompt=prompt,
        pattern="(\{.*?\})",
        options={"format": "json", "temperature": 0.2},
    )
    cluster_topic = literal_eval(topics_str)
    prediction, _ = cluster_topic.popitem()

    with open(results_path, "a") as f:
        f.write(f"{index},{row['topic']},{int(prediction)}\n")

In [ ]:
df_results = pd.read_csv(results_path)

print(df_results['class'].value_counts())

show_results(df_results)

## HSMVideoSumm processed

In [ ]:
df = pd.read_csv((PROJECT_DIR / "data/datasets/hsmvideosumm/hsmvideosumm_processed.csv"), sep=";")
topics = {
    "0": "objetivo",
    "1": "subjetivo",
}
df["topic"] = df["subjetividade"].map({v: k for k, v in topics.items()})
df = df[["topic", "content"]]
df = df.sample(frac=1)
df.head()

In [ ]:
results_path = (PROJECT_DIR / "data/datasets/hsmvideosumm/results_processed.csv")
with open(results_path, "w") as f:
    f.write("index,class,prediction\n")

prompts_template = VideoSegmenterPrompts()
llm_adapter = OllamaAdapter()

for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Classifying content"):
    prompt = prompts_template.classify_subtopic.format(
        content=row["content"], topics=topics
    )

    topics_str = llm_adapter.generate_pattern(
        prompt=prompt,
        pattern="(\{.*?\})",
        options={"format": "json", "temperature": 0.2},
    )
    cluster_topic = literal_eval(topics_str)
    prediction, _ = cluster_topic.popitem()

    with open(results_path, "a") as f:
        f.write(f"{index},{row['topic']},{int(prediction)}\n")

In [ ]:
df_results = pd.read_csv(results_path)

print(df_results['class'].value_counts())

show_results(df_results)

## HSMVideoSumm groundtruth

In [ ]:
data = []
for file_path in Path((PROJECT_DIR / "data/datasets/hsmvideosumm")).rglob("*.json"):
    data.extend(json.load(open(file_path, "r")))
df = pd.DataFrame(data)

values_remap = {
    "Neutro": "Neutro",
    "Neutro - sem faces": "Neutro",
    "Entrevista": "Entrevista",
    "Opinião de repórter": "Opinião de repórter",
    "Opinião de repórter/entrevista": "Opinião de repórter",
}
df["topic"] = df["class"].map(values_remap)

topics = {
    "0": "Neutro",
    "1": "Entrevista",
    "2": "Opinião de repórter",
}
df["topic"] = df["topic"].map({v: k for k, v in topics.items()})

df = df[["topic", "content"]]
df = df.sample(frac=1)
df.head()

In [ ]:
results_path = (PROJECT_DIR / "data/datasets/hsmvideosumm/results_groundtruth.csv")
with open(results_path, "w") as f:
    f.write("index,class,prediction\n")

prompts_template = VideoSegmenterPrompts()
llm_adapter = OllamaAdapter()

for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Classifying content"):
    prompt = prompts_template.classify_subtopic.format(
        content=row["content"], topics=topics
    )

    topics_str = llm_adapter.generate_pattern(
        prompt=prompt,
        pattern="(\{.*?\})",
        options={"format": "json", "temperature": 0.2},
    )
    cluster_topic = literal_eval(topics_str)
    prediction, _ = cluster_topic.popitem()

    with open(results_path, "a") as f:
        f.write(f"{index},{row['topic']},{int(prediction)}\n")

In [ ]:
df_results = pd.read_csv(results_path)

print(df_results['class'].value_counts())

show_results(df_results)

# Subtopic generation

In [ ]:
prompt = """Evaluate the quality of the subtopics generated for a video transcription. Rate the list of subtopics on four dimensions on a scale from 1 (worst) to 5 (best):

1. Relevance: How well the subtopics list match the actual semantic content in transcript.
Ensures the subtopics are grounded in the source material, not hallucinated.
Score 1 (worst): Unrelated or misleading.    
Score 5 (best): Perfectly matches the content, no extraneous concepts.
    
2. Specificity: Whether the subtopics list are at an appropriate level of detail, not too broad, not too narrow.
Broad topics lose nuance; overly specific topics fragment coherence.
Score 1 (worst): Too vague or too hyper-specific.
Score 5 (best): Accurately scoped.

3. Coverage: How well the subtopics list capture the main ideas of the video, without omitting key points.
Prevents missing critical information that affects the meaning.
Score 1 (worst): Misses most important concepts.
Score 5 (best): Includes all key elements.

4. Distinctiveness: How well the subtopics list are differentiated from each other.
Distinct subtopics improve diversity and avoid repetition.
Score 1 (worst): Nearly identical subtopics.
Score 5 (best): Subtopics are clearly different from each other .

Video transcription:
{video_transcription}

Subtopics:
{subtopics}


Your output should be a JSON object with the following structure:
{{
    "relevance": <score from 1 to 5>,
    "specificity": <score from 1 to 5>,
    "coverage": <score from 1 to 5>,
    "distinctiveness": <score from 1 to 5>
}}

"""

In [ ]:
from open_video_summary.parsers.video import VideoLoader

videos = VideoLoader.load_videos_from_json((PROJECT_DIR / "data/processed/bebe_real.json"))

data = []
for video in videos:
    print(f"Evaluating video: {video.name}")

    transcription = " ".join(s.content for s in video.segments)
    prompt_filled = prompt.format(
        video_transcription=transcription,
        subtopics="\n".join(video.topics)
    )
    llm_judge_adapter = OllamaAdapter(model="gpt-oss:20b")
    
    topics_str = llm_judge_adapter.generate_pattern(
        prompt=prompt_filled,
        pattern="(\{.*?\})",
        options={"format": "json", "temperature": 0.2},
    )
    evaluation = literal_eval(topics_str)
    data.append((video, evaluation))

In [ ]:
# GPT-oss
data

In [ ]:
# Deepseek
data